# Create a unique text-only RAG benchmark

This notebook selects:

- **10 relevant PDFs**
- **4 unique extractive text-only questions per relevant PDF**
- **4 unique abstractive text-only questions per relevant PDF**
- **8 questions per relevant PDF**
- **80 total evaluation questions**
  - 40 extractive
  - 40 abstractive
- **5 irrelevant/distractor PDFs**
- **No questions from distractor PDFs**
- **15 PDFs total**

The filter is strict:

```python
source == "text"
```

Questions whose source is `text-image`, `text-table`, or `text-table-image` are excluded.

Uniqueness is enforced using both:

- `query_id`
- normalized question text

So the final evaluation set contains no repeated question IDs or duplicate question wording.


## Files required

Put these files beside the notebook:

```text
Eval_data.csv
All_PDFs.csv
pdfs-20250915T225624Z-1-001.zip
pdfs-20250915T225624Z-1-002.zip
```

The notebook scans both ZIP files but extracts only the selected 15 PDFs.


In [2]:
# Install only if needed.
%pip install pandas requests

from pathlib import Path
import shutil
import zipfile

import pandas as pd
import requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 9.8 MB/s eta 0:00:000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 10.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 8.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 21.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 kB 7.6 MB/s eta 0:00:00

[notice] A new release of pip available: 22.3.1 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# -------------------------
# Configuration
# -------------------------

DATA_DIR = Path(".")
EVAL_CSV = DATA_DIR / "Eval_data.csv"
ALL_PDFS_CSV = DATA_DIR / "All_PDFs.csv"

OUTPUT_DIR = DATA_DIR / "text_only_rag_benchmark_4x4_unique"
RELEVANT_PDF_DIR = OUTPUT_DIR / "pdfs" / "relevant"
DISTRACTOR_PDF_DIR = OUTPUT_DIR / "pdfs" / "distractors"
METADATA_DIR = OUTPUT_DIR / "metadata"

RANDOM_SEED = 42
N_RELEVANT_PDFS = 10
EXTRACTIVE_PER_PDF = 4
ABSTRACTIVE_PER_PDF = 4
N_DISTRACTORS = 5

# Avoid selecting unusually large distractors for this small benchmark.
MAX_DISTRACTOR_SIZE_KB = 10_000

ZIP_FILES = sorted(DATA_DIR.glob("pdfs-*.zip"))

for folder in [
    RELEVANT_PDF_DIR,
    DISTRACTOR_PDF_DIR,
    METADATA_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("ZIP files found:")
for zip_path in ZIP_FILES:
    print(" -", zip_path)


ZIP files found:
 - pdfs-20250915T225624Z-1-001.zip
 - pdfs-20250915T225624Z-1-002.zip


In [4]:
# -------------------------
# Load and validate the CSV files
# -------------------------

eval_df = pd.read_csv(EVAL_CSV)
all_pdfs_df = pd.read_csv(ALL_PDFS_CSV)

required_eval_columns = {
    "query_id",
    "query",
    "answer",
    "document_name",
    "pdf_filename",
    "pdf_url",
    "type",
    "source",
}

required_pdf_columns = {
    "pdf_filename",
    "pdf_url",
    "file_exists",
    "file_size_kb",
    "document_type",
}

missing_eval = required_eval_columns - set(eval_df.columns)
missing_pdfs = required_pdf_columns - set(all_pdfs_df.columns)

if missing_eval:
    raise ValueError(
        f"Eval_data.csv is missing columns: {sorted(missing_eval)}"
    )

if missing_pdfs:
    raise ValueError(
        f"All_PDFs.csv is missing columns: {sorted(missing_pdfs)}"
    )

print("Eval_data shape:", eval_df.shape)
print("All_PDFs shape:", all_pdfs_df.shape)


Eval_data shape: (3045, 14)
All_PDFs shape: (1001, 9)


In [5]:
# -------------------------
# Keep only text-only extractive and abstractive questions
# -------------------------

text_queries = eval_df[
    eval_df["source"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("text")
    & eval_df["type"].isin(["extractive", "abstractive"])
].copy()

# Normalize wording so questions that differ only by capitalization
# or repeated whitespace count as duplicates.
text_queries["query_normalized"] = (
    text_queries["query"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# Remove duplicate IDs and duplicate normalized question wording globally.
unique_queries = (
    text_queries
    .sort_values(["pdf_filename", "type", "query_id"])
    .drop_duplicates(subset=["query_id"], keep="first")
    .drop_duplicates(subset=["query_normalized"], keep="first")
    .reset_index(drop=True)
)

print("Text-only rows before deduplication:", len(text_queries))
print("Unique text-only rows:", len(unique_queries))

assert unique_queries["query_id"].is_unique
assert unique_queries["query_normalized"].is_unique
assert unique_queries["source"].str.lower().eq("text").all()


Text-only rows before deduplication: 1914
Unique text-only rows: 1913


In [6]:
# -------------------------
# Find PDFs that have at least 4 unique questions of each type
# -------------------------

question_counts = (
    unique_queries
    .groupby(["pdf_filename", "type"])
    .size()
    .unstack(fill_value=0)
)

eligible_pdfs = question_counts[
    (question_counts.get("extractive", 0) >= EXTRACTIVE_PER_PDF)
    & (
        question_counts.get("abstractive", 0)
        >= ABSTRACTIVE_PER_PDF
    )
].copy()

print("Eligible relevant PDFs:", len(eligible_pdfs))
display(eligible_pdfs)

if len(eligible_pdfs) < N_RELEVANT_PDFS:
    raise ValueError(
        f"Only {len(eligible_pdfs)} PDFs have at least "
        f"{EXTRACTIVE_PER_PDF} unique extractive and "
        f"{ABSTRACTIVE_PER_PDF} unique abstractive "
        f"text-only questions."
    )


Eligible relevant PDFs: 19


type,abstractive,extractive
pdf_filename,,
2403.03363v6.pdf,4,4
2403.05821v2.pdf,4,4
2403.13015v2.pdf,4,4
2404.18137v2.pdf,4,6
2406.17972v3.pdf,4,5
2406.18052v3.pdf,4,4
2407.07009v2.pdf,4,4
2407.16205v5.pdf,4,5
2408.06427v2.pdf,5,5


In [7]:
# -------------------------
# Choose 10 unique relevant PDFs
# -------------------------

selected_pdf_names = (
    eligible_pdfs
    .sample(
        n=N_RELEVANT_PDFS,
        random_state=RANDOM_SEED,
    )
    .index.tolist()
)

assert len(selected_pdf_names) == N_RELEVANT_PDFS
assert len(set(selected_pdf_names)) == N_RELEVANT_PDFS

print("Selected relevant PDFs:")
for filename in selected_pdf_names:
    print(" -", filename)


Selected relevant PDFs:
 - 2403.03363v6.pdf
 - 2406.18052v3.pdf
 - 2408.14507v2.pdf
 - 2403.05821v2.pdf
 - 2408.06427v2.pdf
 - 2411.12375v3.pdf
 - 2404.18137v2.pdf
 - 2410.12036v2.pdf
 - 2410.23587v3.pdf
 - 2412.11692v4.pdf


In [8]:
# -------------------------
# Select 4 unique extractive and 4 unique abstractive
# questions for every relevant PDF
# -------------------------

selected_query_frames = []

for index, pdf_filename in enumerate(selected_pdf_names):
    pdf_queries = unique_queries[
        unique_queries["pdf_filename"].eq(pdf_filename)
    ]

    extractive_queries = pdf_queries[
        pdf_queries["type"].eq("extractive")
    ].sample(
        n=EXTRACTIVE_PER_PDF,
        random_state=RANDOM_SEED + index,
    )

    abstractive_queries = pdf_queries[
        pdf_queries["type"].eq("abstractive")
    ].sample(
        n=ABSTRACTIVE_PER_PDF,
        random_state=RANDOM_SEED + 100 + index,
    )

    selected_query_frames.extend(
        [extractive_queries, abstractive_queries]
    )

selected_queries = pd.concat(
    selected_query_frames,
    ignore_index=True,
)

# These checks ensure uniqueness across the entire final benchmark,
# not only within one PDF.
assert selected_queries["query_id"].is_unique
assert selected_queries["query_normalized"].is_unique
assert len(selected_queries) == 80
assert selected_queries["type"].eq("extractive").sum() == 40
assert selected_queries["type"].eq("abstractive").sum() == 40
assert selected_queries["source"].str.lower().eq("text").all()

distribution = (
    selected_queries
    .groupby(["pdf_filename", "type"])
    .size()
    .unstack(fill_value=0)
)

display(distribution)

assert distribution["extractive"].eq(4).all()
assert distribution["abstractive"].eq(4).all()


type,abstractive,extractive
pdf_filename,,
2403.03363v6.pdf,4,4
2403.05821v2.pdf,4,4
2404.18137v2.pdf,4,4
2406.18052v3.pdf,4,4
2408.06427v2.pdf,4,4
2408.14507v2.pdf,4,4
2410.12036v2.pdf,4,4
2410.23587v3.pdf,4,4
2411.12375v3.pdf,4,4


In [9]:
# -------------------------
# Select 5 unique distractor PDFs
#
# No queries are selected from distractor PDFs.
# -------------------------

distractor_candidates = (
    all_pdfs_df[
        all_pdfs_df["document_type"].eq("distractor")
        & all_pdfs_df["file_exists"].eq(True)
        & all_pdfs_df["file_size_kb"].between(
            100,
            MAX_DISTRACTOR_SIZE_KB,
        )
        & ~all_pdfs_df["pdf_filename"].isin(selected_pdf_names)
    ]
    .drop_duplicates(subset=["pdf_filename"])
)

if len(distractor_candidates) < N_DISTRACTORS:
    raise ValueError("Not enough unique distractor PDFs were found.")

selected_distractors = distractor_candidates.sample(
    n=N_DISTRACTORS,
    random_state=RANDOM_SEED,
).reset_index(drop=True)

assert selected_distractors["pdf_filename"].is_unique
assert len(selected_distractors) == 5
assert not set(selected_distractors["pdf_filename"]).intersection(
    selected_pdf_names
)

display(selected_distractors)


,doc_id,arxiv_id,pdf_filename,pdf_url,file_exists,file_size_kb,query_count,sample_query,document_type
0,2404.05230v2,Distractor:2404.05230v2,2404.05230v2.pdf,N/A (Distractor),True,1229.7,0,N/A (Distractor),distractor
1,2409.12067v4,Distractor:2409.12067v4,2409.12067v4.pdf,N/A (Distractor),True,3195.6,0,N/A (Distractor),distractor
2,2409.04233v1,Distractor:2409.04233v1,2409.04233v1.pdf,N/A (Distractor),True,440.1,0,N/A (Distractor),distractor
3,2412.20173v3,Distractor:2412.20173v3,2412.20173v3.pdf,N/A (Distractor),True,201.9,0,N/A (Distractor),distractor
4,2411.10956v2,Distractor:2411.10956v2,2411.10956v2.pdf,N/A (Distractor),True,489.3,0,N/A (Distractor),distractor


In [10]:
# -------------------------
# Create and save metadata files
# -------------------------

selected_relevant_documents = (
    selected_queries[
        ["pdf_filename", "pdf_url", "document_name"]
    ]
    .drop_duplicates(subset=["pdf_filename"])
    .reset_index(drop=True)
)

queries_to_save = selected_queries.drop(
    columns=["query_normalized"]
)

queries_path = METADATA_DIR / "evaluation_queries.csv"
relevant_path = METADATA_DIR / "relevant_documents.csv"
distractors_path = METADATA_DIR / "distractor_documents.csv"

queries_to_save.to_csv(queries_path, index=False)
selected_relevant_documents.to_csv(relevant_path, index=False)
selected_distractors.to_csv(distractors_path, index=False)

print("Saved:")
print(" -", queries_path)
print(" -", relevant_path)
print(" -", distractors_path)


Saved:
 - text_only_rag_benchmark_4x4_unique/metadata/evaluation_queries.csv
 - text_only_rag_benchmark_4x4_unique/metadata/relevant_documents.csv
 - text_only_rag_benchmark_4x4_unique/metadata/distractor_documents.csv


In [11]:
# -------------------------
# Build a filename-to-ZIP lookup
# -------------------------

def build_zip_pdf_index(
    zip_paths: list[Path],
) -> dict[str, tuple[Path, str]]:
    pdf_index = {}

    for zip_path in zip_paths:
        print("Scanning:", zip_path)

        with zipfile.ZipFile(zip_path, "r") as archive:
            for member_name in archive.namelist():
                filename = Path(member_name).name

                if filename.lower().endswith(".pdf"):
                    pdf_index.setdefault(
                        filename,
                        (zip_path, member_name),
                    )

    return pdf_index


zip_pdf_index = build_zip_pdf_index(ZIP_FILES)
print("PDFs found inside ZIP files:", len(zip_pdf_index))


Scanning: pdfs-20250915T225624Z-1-001.zip
Scanning: pdfs-20250915T225624Z-1-002.zip
PDFs found inside ZIP files: 1001


In [12]:
# -------------------------
# PDF extraction/download helpers
# -------------------------

def extract_pdf_from_zip(
    pdf_filename: str,
    destination_dir: Path,
    zip_index: dict[str, tuple[Path, str]],
) -> Path | None:
    match = zip_index.get(pdf_filename)

    if match is None:
        return None

    zip_path, member_name = match
    destination_path = destination_dir / pdf_filename

    with zipfile.ZipFile(zip_path, "r") as archive:
        with archive.open(member_name) as source:
            with destination_path.open("wb") as destination:
                shutil.copyfileobj(source, destination)

    return destination_path


def download_pdf(
    pdf_url: str,
    pdf_filename: str,
    destination_dir: Path,
) -> Path:
    if not isinstance(pdf_url, str) or not pdf_url.startswith("http"):
        raise ValueError(
            f"No usable download URL exists for {pdf_filename}."
        )

    destination_path = destination_dir / pdf_filename

    response = requests.get(
        pdf_url,
        timeout=60,
        allow_redirects=True,
    )
    response.raise_for_status()

    content_type = response.headers.get(
        "content-type",
        "",
    ).lower()

    if "pdf" not in content_type and not response.content.startswith(b"%PDF"):
        raise ValueError(
            f"The response for {pdf_filename} is not a PDF."
        )

    destination_path.write_bytes(response.content)
    return destination_path


In [13]:
# -------------------------
# Extract/download the 10 relevant PDFs
# -------------------------

relevant_results = []

for row in selected_relevant_documents.itertuples(index=False):
    extracted_path = extract_pdf_from_zip(
        pdf_filename=row.pdf_filename,
        destination_dir=RELEVANT_PDF_DIR,
        zip_index=zip_pdf_index,
    )

    if extracted_path is not None:
        relevant_results.append(
            {
                "pdf_filename": row.pdf_filename,
                "status": "extracted_from_zip",
                "path": str(extracted_path),
            }
        )
    else:
        downloaded_path = download_pdf(
            pdf_url=row.pdf_url,
            pdf_filename=row.pdf_filename,
            destination_dir=RELEVANT_PDF_DIR,
        )

        relevant_results.append(
            {
                "pdf_filename": row.pdf_filename,
                "status": "downloaded",
                "path": str(downloaded_path),
            }
        )

display(pd.DataFrame(relevant_results))


,pdf_filename,status,path
0,2403.03363v6.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/releva...
1,2406.18052v3.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/releva...
2,2408.14507v2.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/releva...
3,2403.05821v2.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/releva...
4,2408.06427v2.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/releva...
5,2411.12375v3.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/releva...
6,2404.18137v2.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/releva...
7,2410.12036v2.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/releva...
8,2410.23587v3.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/releva...
9,2412.11692v4.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/releva...


In [14]:
# -------------------------
# Extract the 5 distractor PDFs
# -------------------------

distractor_results = []
missing_distractors = []

for row in selected_distractors.itertuples(index=False):
    extracted_path = extract_pdf_from_zip(
        pdf_filename=row.pdf_filename,
        destination_dir=DISTRACTOR_PDF_DIR,
        zip_index=zip_pdf_index,
    )

    if extracted_path is None:
        missing_distractors.append(row.pdf_filename)
    else:
        distractor_results.append(
            {
                "pdf_filename": row.pdf_filename,
                "status": "extracted_from_zip",
                "path": str(extracted_path),
            }
        )

if missing_distractors:
    raise FileNotFoundError(
        "These distractors were not found in the ZIP files: "
        + ", ".join(missing_distractors)
    )

display(pd.DataFrame(distractor_results))


,pdf_filename,status,path
0,2404.05230v2.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/distra...
1,2409.12067v4.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/distra...
2,2409.04233v1.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/distra...
3,2412.20173v3.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/distra...
4,2411.10956v2.pdf,extracted_from_zip,text_only_rag_benchmark_4x4_unique/pdfs/distra...


In [15]:
# -------------------------
# Final verification
# -------------------------

relevant_pdf_files = sorted(RELEVANT_PDF_DIR.glob("*.pdf"))
distractor_pdf_files = sorted(DISTRACTOR_PDF_DIR.glob("*.pdf"))

assert len(relevant_pdf_files) == 10
assert len(distractor_pdf_files) == 5
assert len(selected_queries) == 80
assert selected_queries["query_id"].is_unique
assert selected_queries["query_normalized"].is_unique
assert selected_queries["type"].eq("extractive").sum() == 40
assert selected_queries["type"].eq("abstractive").sum() == 40
assert selected_queries["source"].str.lower().eq("text").all()

print("Benchmark created successfully.")
print("Relevant PDFs:", len(relevant_pdf_files))
print("Distractor PDFs:", len(distractor_pdf_files))
print("Unique evaluation questions:", len(selected_queries))
print("Extractive questions:", selected_queries["type"].eq("extractive").sum())
print("Abstractive questions:", selected_queries["type"].eq("abstractive").sum())
print("Output folder:", OUTPUT_DIR.resolve())


Benchmark created successfully.
Relevant PDFs: 10
Distractor PDFs: 5
Unique evaluation questions: 80
Extractive questions: 40
Abstractive questions: 40
Output folder: /Users/aryan/Downloads/Extract PDFs RAG/text_only_rag_benchmark_4x4_unique


## Output

```text
text_only_rag_benchmark_4x4_unique/
├── pdfs/
│   ├── relevant/                  # 10 PDFs
│   └── distractors/               # 5 PDFs
└── metadata/
    ├── evaluation_queries.csv     # 80 unique questions
    ├── relevant_documents.csv
    └── distractor_documents.csv
```
